# 01 — Menu API ingestion

This notebook begins the menu recommendation prototype described in `DESIGN.md`.

The first step is deliberately small: fetch menu items and categories from two API endpoints without assuming a specific provider's response schema. The returned payload remains raw so source-specific normalization can be added as the next layer.

## Imports

Only Python's standard library is used, so this first step has no third-party dependencies.

In [1]:
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Any, Mapping
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen
import json

## API result and errors

`ApiMenuPayload` keeps the two source documents together and records when they were retrieved. `MenuApiError` provides a single error type for callers while retaining the original exception as its cause.

In [2]:
class MenuApiError(RuntimeError):
    """Raised when menu data cannot be retrieved or decoded."""


@dataclass(frozen=True)
class ApiMenuPayload:
    menu: Any
    categories: Any
    menu_endpoint: str
    categories_endpoint: str
    fetched_at: datetime

## HTTP JSON helper

The helper performs a `GET`, requests JSON, checks the HTTP response, and decodes UTF-8 JSON. Query parameters are encoded rather than concatenated into the URL. Authentication can be supplied through the `headers` argument of the public function.

In [3]:
def _with_query_params(
    endpoint: str,
    params: Mapping[str, Any] | None = None,
) -> str:
    if not params:
        return endpoint

    separator = "&" if "?" in endpoint else "?"
    return f"{endpoint}{separator}{urlencode(params, doseq=True)}"


def _get_json(
    endpoint: str,
    *,
    headers: Mapping[str, str] | None = None,
    params: Mapping[str, Any] | None = None,
    timeout_seconds: float = 30.0,
) -> Any:
    if timeout_seconds <= 0:
        raise ValueError("timeout_seconds must be greater than zero")

    url = _with_query_params(endpoint, params)
    request_headers = {"Accept": "application/json"}
    if headers:
        request_headers.update(headers)

    request = Request(url, headers=request_headers, method="GET")

    try:
        with urlopen(request, timeout=timeout_seconds) as response:
            charset = response.headers.get_content_charset() or "utf-8"
            body = response.read().decode(charset)
            return json.loads(body)
    except HTTPError as exc:
        raise MenuApiError(
            f"GET {endpoint} failed with HTTP status {exc.code}"
        ) from exc
    except URLError as exc:
        raise MenuApiError(f"GET {endpoint} failed: {exc.reason}") from exc
    except (UnicodeDecodeError, json.JSONDecodeError) as exc:
        raise MenuApiError(
            f"GET {endpoint} did not return valid JSON"
        ) from exc

## Fetch menu items and categories

This is the public function for the first iteration. Menu and category query parameters are separate because APIs often paginate or filter these resources differently. Shared headers can include an API key or bearer token; credentials should be loaded from environment variables or a secret manager rather than written into the notebook.

In [4]:
def read_menu_from_api(
    menu_endpoint: str,
    categories_endpoint: str,
    *,
    headers: Mapping[str, str] | None = None,
    menu_params: Mapping[str, Any] | None = None,
    category_params: Mapping[str, Any] | None = None,
    timeout_seconds: float = 30.0,
) -> ApiMenuPayload:
    """Read raw menu and category JSON documents from API endpoints.

    The payloads are intentionally not normalized here. A source adapter will
    map them into the canonical Menu model after the API schema is known.
    """
    if not menu_endpoint.strip():
        raise ValueError("menu_endpoint must not be empty")
    if not categories_endpoint.strip():
        raise ValueError("categories_endpoint must not be empty")

    menu = _get_json(
        menu_endpoint,
        headers=headers,
        params=menu_params,
        timeout_seconds=timeout_seconds,
    )
    categories = _get_json(
        categories_endpoint,
        headers=headers,
        params=category_params,
        timeout_seconds=timeout_seconds,
    )

    return ApiMenuPayload(
        menu=menu,
        categories=categories,
        menu_endpoint=menu_endpoint,
        categories_endpoint=categories_endpoint,
        fetched_at=datetime.now(timezone.utc),
    )

## Usage template

Replace the example URLs and adapt the headers and parameters to the actual API. The call is commented out so running all cells does not make an accidental external request.

In [5]:
import os


menu_endpoint = "https://ordering-service-prod-api-gateway-bcjbztjy.nw.gateway.dev/api/v1/store/08a065b2-7748-4d49-99e6-ece88e6038f9/menu/items"
category_endpoint="https://ordering-service-prod-api-gateway-bcjbztjy.nw.gateway.dev/api/v1/store/08a065b2-7748-4d49-99e6-ece88e6038f9/menu/categories"

payload = read_menu_from_api(
    menu_endpoint=menu_endpoint,
    categories_endpoint=category_endpoint,
    # headers={"Authorization": f"Bearer {os.environ['MENU_API_TOKEN']}"},
    menu_params={},
    category_params={},
)
#
print(f"Fetched at: {payload.fetched_at.isoformat()}")
print(json.dumps(payload.categories, indent=2)[:2_000])
print(json.dumps(payload.menu, indent=2)[:2_000])

Fetched at: 2026-10-05T13:10:16.557692+00:00
[
  {
    "id": "0ebf4875-ea06-4f54-9149-ac060fedcd64",
    "name": "Roses",
    "description": "",
    "image": "",
    "availability": "AVAILABLE",
    "menu_item_order": [
      "6acc2f74-92ef-4ac7-9d08-f48217524a65",
      "ab1ac371-7f2b-4380-b656-b7139473f786",
      "d59dfa4a-7515-4c94-8f1f-3e52736f372c",
      "8c01995c-2d80-4cd2-8e41-9f0c7d03de0c",
      "aa88011f-8e69-4ce3-a778-2d32f6abf82a",
      "d0538aec-5a9d-47ae-a82a-d7721d7bb53f",
      "34343ca2-bc28-4e63-935c-0134ddf6f249"
    ],
    "subgroups_ordered": [],
    "child_category": false,
    "style": {
      "textColour": "",
      "backgroundColour": ""
    },
    "printer_config": {
      "id": "7228510c-8ab3-4ffd-a79f-b163d5787f59",
      "name": "BAR",
      "ip": "192.168.1.38",
      "pos_id": "111bef5a3c762496",
      "pos_name": "Main POS",
      "store_device_id": "2a68d7ca-b3ad-4e8f-9fcd-d84b4fe5a4bb",
      "device_id": null,
      "printer_type": "NETWORK",
     

## Normalize the API response

The adapter normalizes channel-specific base prices, sizes, and custom variation groups. Product-based groups and sub-product fields are ignored for now. It uses `Decimal` for exact prices, retains source identifiers, and treats missing allergen data as no allergens. Presentation-only subgroup and ordering fields are ignored because retrieval will use item-level embeddings.

In [6]:
from menu_normalization import normalize_api_menu

menu = normalize_api_menu(
    payload,
    menu_id="1c037ca1-b41a-4482-b9c7-85ecee7ab3e1",
    restaurant_name="Example restaurant",  # Replace with the store name.
    currency="GBP",
    channel="ONLINE_DELIVERY",
)

print(f"Normalized {len(menu.items)} items in {len(menu.categories)} categories")
print(menu.items[0])

Normalized 224 items in 22 categories
MenuItem(id='ef02fbf0-0691-48d1-a080-9d2f7c0fcf1d', name='Blanco Candidato 250ml', description=None, category_ids=('69bd9477-d325-4687-9d03-f9e05f49169f',), base_price=Decimal('6.75'), currency='GBP', variants=(), ingredients=None, allergens=(), dietary_attributes=(), semantic_attributes=(), availability=Availability(status='unavailable', checked_at=datetime.datetime(2026, 10, 5, 13, 10, 16, 557692, tzinfo=datetime.timezone.utc)), source_reference=SourceReference(source_type='api', source_id='https://ordering-service-prod-api-gateway-bcjbztjy.nw.gateway.dev/api/v1/store/08a065b2-7748-4d49-99e6-ece88e6038f9/menu/items', external_record_id='ef02fbf0-0691-48d1-a080-9d2f7c0fcf1d'), embedding_text='Blanco Candidato 250ml\nWhites', variation_groups=(), inferred_description=None)


### Normalization notes

- Choose the channel used by the recommendation surface; prices genuinely differ across channels.
- Items with sizes have fully priced `ItemVariant` records and no `base_price`.
- `subgroups_ordered`, `subgroup_id`, `menu_item_order`, `variation_order`, and size `position` are presentation concerns and are intentionally ignored.
- `course` is printer-routing metadata and is intentionally ignored.
- Missing, null, or empty allergen data is normalized to an empty tuple and treated as no allergens.
- Only `CUSTOM` entries in `menu_item_variations` become `variation_groups`, with names, availability, required/optional status, selection limits, and choice price adjustments. Choice names are included in embedding text. `PRODUCT_BASED` groups and all `sub_product_selection` fields are ignored for now.

## Validate before embedding

Validation is non-mutating. Errors block the next ingestion stage; warnings identify usable records that may need attention.

In [7]:
from menu_validation import validate_menu

validation = validate_menu(menu)
print(
    f"Validation: {len(validation.errors)} error(s), "
    f"{len(validation.warnings)} warning(s)"
)
for issue in (*validation.errors, *validation.warnings):
    print(f"[{issue.code}] {issue.message}")

validation.raise_for_errors()

Validation: 0 error(s), 0 warning(s)
